In [12]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion9-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

-------------------------------------------
Batch: 772
-------------------------------------------
+------------------------+---------+------+-----------------------+--------------+--------------------+-----------+-------+-------+-------------+-------------+-------+-------------+---------+
|topic                   |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId            |temperatura|humedad|presion|origen       |timestamp    |isValid|processedAt  |latencyMs|
+------------------------+---------+------+-----------------------+--------------+--------------------+-----------+-------+-------+-------------+-------------+-------+-------------+---------+
|sensores-fisicos-eventos|2        |14775 |2026-10-15 17:29:35.829|sensor.lectura|esp32-fisico-lab    |19.6       |20.0   |1013.2 |hardware_real|1792085375508|true   |1792085380005|4497     |
|sensores-fisicos-eventos|2        |14776 |2026-10-15 17:29:37.571|sensor.lectura|esp32-fisico-lab    |14.6       |32.1   |1013.2 |ha

In [13]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "sensores-fisicos-eventos")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(
        col("topic"), col("partition"), col("offset"),
        col("timestamp").alias("kafkaTimestamp"),
        col("value").cast("string").alias("json_str"),
    )
    .select(
        "topic", "partition", "offset", "kafkaTimestamp",
        from_json(col("json_str"), esquema_evento).alias("e"),
    )
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

observables = (
    eventos
    .withColumn(
        "isValid",
        col("tipoEvento").isNotNull()
        & col("sensorId").isNotNull()
        & col("temperatura").isNotNull()
        & col("timestamp").isNotNull(),
    )
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)
observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)

-------------------------------------------
Batch: 774
-------------------------------------------
+------------------------+---------+------+-----------------------+--------------+--------------------+-----------+-------+-------+-------------+-------------+-------+-------------+---------+
|topic                   |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId            |temperatura|humedad|presion|origen

In [14]:
# Stream 1: Visualización en consola
consulta = (
    observables.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

# Stream 2: Persistencia en Parquet
ARTIFACTS = "./artifacts/sensores_observabilidad"
CHECKPOINT_PARQUET = "./artifacts/chk_obs_sensores"

consulta2 = (
    observables.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+
|topic|partition|offset|kafkaTimestamp|tipoEvento|sensorId|temperatura|humedad|presion|origen|timestamp|isValid|processedAt|latencyMs|
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+

-------------------------------------------
Batch: 775
-------------------------------------------
+------------------------+---------+------+-----------------------+--------------+--------------------+-----------+-------+-------+-------------+-------------+-------+-------------+---------+
|topic                   |partition|offset|kafkaTimestamp         |tipo

In [18]:
import json

progreso = consulta.lastProgress

if progreso is None:
    print("Esperando micro-lote... corre la celda en 5 segundos.")
else:
    fuente = progreso["sources"][0]
    metricas = fuente.get("metrics", {})
    estado = "idle" if progreso["numInputRows"] == 0 else "processed"

    log = {
        "service": "spark-streaming",
        "component": "observabilidad",
        "topic": "sensores-fisicos-eventos",
        "batchId": progreso["batchId"],
        "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": progreso["inputRowsPerSecond"],
        "processedRowsPerSecond": progreso["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
        "status": estado
    }
    print(json.dumps(log, indent=2))

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "sensores-fisicos-eventos",
  "batchId": 62,
  "numInputRows": 8,
  "inputRowsPerSecond": 1.6,
  "processedRowsPerSecond": 13.1,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}
-------------------------------------------
Batch: 63
-------------------------------------------
-------------------------------------------
Batch: 837
-------------------------------------------
+------------------------+---------+------+-----------------------+--------------+--------------------+-----------+-------+-------+-------------+-------------+-------+-------------+---------+
|topic                   |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId            |temperatura|humedad|presion|origen       |timestamp    |isValid|processedAt  |latencyMs|
+------------------------+---------+------+-----------------------+--------------+--------------------+-----------+---

In [19]:
# 1. Detener streams activos de forma segura
consulta.stop()
try:
    consulta2.stop()
except:
    pass

# 2. Análisis de latencia agregada por ventana de 1 minuto (Sección 3.9 oficial)
from pyspark.sql.functions import col, window, avg, min as spark_min, max as spark_max, count

ARTIFACTS = "./artifacts/sensores_observabilidad"
guardado = spark.read.parquet(ARTIFACTS)

latencia_por_minuto = (
    guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(
        count("*").alias("numEventos"),
        avg("latencyMs").alias("avgLatencyMs"),
        spark_min("latencyMs").alias("minLatencyMs"),
        spark_max("latencyMs").alias("maxLatencyMs"),
    )
    .select(
        col("window.start").alias("inicioVentana"),
        col("window.end").alias("finVentana"),
        "numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs",
    )
    .orderBy("inicioVentana")
)
latencia_por_minuto.show(20, truncate=False)

+-------------------+-------------------+----------+------------------+------------+------------+
|inicioVentana      |finVentana         |numEventos|avgLatencyMs      |minLatencyMs|maxLatencyMs|
+-------------------+-------------------+----------+------------------+------------+------------+
|2026-10-15 17:29:00|2026-10-15 17:30:00|9         |2553.4444444444443|491         |4419        |
|2026-10-15 17:30:00|2026-10-15 17:31:00|80        |2405.825          |360         |4490        |
|2026-10-15 17:31:00|2026-10-15 17:32:00|80        |2352.1875         |294         |4472        |
|2026-10-15 17:32:00|2026-10-15 17:33:00|80        |2293.75           |223         |4445        |
|2026-10-15 17:33:00|2026-10-15 17:34:00|80        |2239.1375         |158         |4431        |
|2026-10-15 17:34:00|2026-10-15 17:35:00|80        |2183.7125         |87          |4409        |
|2026-10-15 17:35:00|2026-10-15 17:36:00|72        |2107.5694444444443|37          |4387        |
+-------------------